# ML-09 — Validation and Research Claim Audit

**Lane:** Refresh / Content Opportunity Scoring. This notebook audits the actual FlyRank March 2026 research paper and the warehouse-based Logistic Regression developed in ML-08. It does not use the older starter-CSV report referenced by `submission/paper_url.txt`.

## 1. Two paper findings + my methodology questions

I read *The State of AI-Driven SEO* (March 2026), [`docs/flyrank-seo-research-march-2026.pdf`](../../docs/flyrank-seo-research-march-2026.pdf). These questions are intended to clarify how the evidence supports the reported patterns, not to presume that the findings are wrong.

### Finding 1 — Growing and declining content have different profiles (paper p. 6)
The paper reports that its rising-impression group averaged about 3,180 words and 184 days old, compared with about 2,311 words and 230 days for its declining-impression group (74,187 rising and 45,272 falling pages). It describes this as an observational comparison.

**Methodology question:** At what point were word count and page age measured relative to the periods used to classify rising versus declining impressions? Knowing that timing would help distinguish pre-existing page characteristics from changes that may have happened during or after the measured trend. Were missing word counts or differences in page observation history handled consistently between groups?

### Finding 2 — Content performance varies with age, with a 365+ rebound concentrated among refreshed pages (paper p. 7)
The paper reports a health-score peak of 33.1 for ages 61–90 days, a low of 14 for ages 271–365 days, and a 365+ score of 25.1; it says the oldest-bucket rebound is concentrated in refreshed content. The paper cautions that this does not mean age alone reverses decline.

**Methodology question:** Could you clarify the page counts and refresh definition/window behind the 365+ comparison, and whether refreshed pages were compared with similarly aged, non-refreshed pages with similar prior visibility or health? That context would help readers understand how much the observed rebound may reflect which pages were selected for refresh or remained observable, rather than the refresh alone.

These are questions about temporal ordering, cohort composition, and interpretation. The paper’s methodology section describes its evidence as observational, says the machine-learning appendix is exploratory, and notes that it reports no p-values or confidence intervals.

In [1]:
import pandas as pd
from IPython.display import display

paper_review = pd.DataFrame([
    {
        'Paper finding': 'Rising vs. declining profile (p. 6)',
        'Reported comparison': '3,180 vs. 2,311 average words; 184 vs. 230 average age-days',
        'Methodology question': 'Were characteristics measured before the trend window, and were missingness/history comparable?',
    },
    {
        'Paper finding': 'Age curve and 365+ rebound (p. 7)',
        'Reported comparison': 'Health score 33.1 at 61-90 days, 14 at 271-365, 25.1 at 365+',
        'Methodology question': 'What are the bucket counts and refresh window, and is the comparison adjusted for page selection/prior performance?',
    },
])
display(paper_review)

,Paper finding,Reported comparison,Methodology question
0,Rising vs. declining profile (p. 6),"3,180 vs. 2,311 average words; 184 vs. 230 ave...",Were characteristics measured before the trend...
1,Age curve and 365+ rebound (p. 7),"Health score 33.1 at 61-90 days, 14 at 271-365...","What are the bucket counts and refresh window,..."


## 2. My model under an honest split (before/after)

I compare the same ML-08 Logistic Regression and the same ML-07 rule on the same rule-eligible candidates in two ways: (1) a stratified random-row split, which holds out 20% of rows and allows a client to appear on both sides, and (2) a client-held-out split, which holds out 20% of client groups and keeps each client wholly in training or test. The grouped split can therefore have a different row-level test share. Both use seed 42; rows are sorted by pseudonymous entity keys before splitting so the seeded partitions are reproducible. Those keys are used for ordering/grouping only, never as predictors. The target is the April decline proxy; features are March-only.

For each test set, report the test base rate and identical Precision@10/20/50, ROC-AUC, and average precision metrics for the model and the rule. This is a comparison of split designs, not a new model search. A single held-out partition is directional evidence, not a performance guarantee.

In [2]:
from pathlib import Path
import duckdb
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

repo_root = Path.cwd()
while not (repo_root / '.env').is_file() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if not (repo_root / '.env').is_file():
    raise FileNotFoundError('Repository-root .env with HF_TOKEN is required; the token is never displayed.')
hf_token = None
for line in (repo_root / '.env').read_text(encoding='utf-8').splitlines():
    if '=' in line and not line.lstrip().startswith('#'):
        key, value = line.split('=', 1)
        if key.strip() == 'HF_TOKEN':
            hf_token = value.strip().strip(chr(34)).strip(chr(39))
            break
if not hf_token or not hf_token.startswith('hf_'):
    raise ValueError('HF_TOKEN is missing or malformed in .env; token value was not displayed.')
con = duckdb.connect()
con.execute('LOAD httpfs')
con.execute('CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)', [hf_token])
del hf_token
warehouse = 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance'
march_source = 'read_parquet(' + repr(warehouse + '/month=2026-03/*.parquet') + ')'
april_source = 'read_parquet(' + repr(warehouse + '/month=2026-04/*.parquet') + ')'
print('Warehouse access configured from ignored .env; credential was not displayed.')

Warehouse access configured from ignored .env; credential was not displayed.


In [3]:
feature_sql = f'''
WITH march AS (
    SELECT client_hash_id, content_hash_id,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_days_mar,
           SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE) AS impressions_mar,
           SUM(gsc_clicks) FILTER (WHERE gsc_data_available IS TRUE) AS clicks_mar,
           SUM(gsc_sum_position) FILTER (WHERE gsc_data_available IS TRUE) AS position_sum_mar
    FROM {march_source}
    GROUP BY client_hash_id, content_hash_id
), april AS (
    SELECT client_hash_id, content_hash_id,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_days_apr,
           SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE) AS impressions_apr
    FROM {april_source}
    GROUP BY client_hash_id, content_hash_id
)
SELECT march.client_hash_id, march.content_hash_id,
       march.gsc_days_mar, march.impressions_mar, march.clicks_mar,
       march.position_sum_mar / NULLIF(march.impressions_mar, 0) AS avg_position_mar,
       100.0 * march.clicks_mar / NULLIF(march.impressions_mar, 0) AS ctr_mar,
       CAST(april.impressions_apr <= 0.8 * march.impressions_mar AS INTEGER) AS april_decline_label
FROM march INNER JOIN april USING (client_hash_id, content_hash_id)
WHERE march.gsc_days_mar >= 20 AND april.gsc_days_apr >= 20
  AND march.impressions_mar > 0
'''
items = con.execute(feature_sql).df()
con.close()
ID_COLUMNS = ['client_hash_id', 'content_hash_id']
LABEL = 'april_decline_label'
items = items.sort_values(ID_COLUMNS, kind='mergesort').reset_index(drop=True)
assert items[ID_COLUMNS].duplicated().sum() == 0
assert items[LABEL].isin([0, 1]).all()
print(f'Eligible paired client-content items: {len(items):,}')
print(f'April decline-proxy base rate in all eligible items: {items[LABEL].mean():.3f}')
print('One item per pseudonymous client-content pair; both months require at least 20 GSC-available days (IS TRUE).')

Eligible paired client-content items: 95,633
April decline-proxy base rate in all eligible items: 0.495
One item per pseudonymous client-content pair; both months require at least 20 GSC-available days (IS TRUE).


In [4]:
candidate_mask = (
    (items['impressions_mar'] >= 500)
    & (items['avg_position_mar'] > 0)
    & (items['avg_position_mar'] <= 20)
    & (items['ctr_mar'] < 0.5)
)
candidates = items.loc[candidate_mask].copy().reset_index(drop=True)
FEATURES = ['log_impressions_mar', 'log_clicks_mar', 'ctr_mar', 'avg_position_mar', 'gsc_days_mar']
candidates['log_impressions_mar'] = np.log1p(candidates['impressions_mar'])
candidates['log_clicks_mar'] = np.log1p(candidates['clicks_mar'])
assert len(candidates) >= 50
assert candidates[LABEL].nunique() == 2
print(f'ML-07 rule-eligible review pool: {len(candidates):,} items.')
print(f'April decline-proxy base rate within this selected pool: {candidates[LABEL].mean():.3f}')
print('As in ML-08, the tested population is conditional on the ML-07 March-only review rule; results do not cover excluded pages.')

ML-07 rule-eligible review pool: 39,545 items.
April decline-proxy base rate within this selected pool: 0.552
As in ML-08, the tested population is conditional on the ML-07 March-only review rule; results do not cover excluded pages.


In [5]:
SEED = 42
ROW_TEST_SIZE = 0.20
CLIENT_GROUP_TEST_SIZE = 0.20
all_indices = np.arange(len(candidates))
labels = candidates[LABEL].to_numpy(dtype=int)

random_train_idx, random_test_idx = train_test_split(
    all_indices, test_size=ROW_TEST_SIZE, random_state=SEED, stratify=labels,
)
group_splitter = GroupShuffleSplit(n_splits=1, test_size=CLIENT_GROUP_TEST_SIZE, random_state=SEED)
group_train_idx, group_test_idx = next(
    group_splitter.split(candidates[FEATURES], labels, groups=candidates['client_hash_id'])
)

random_train = candidates.iloc[random_train_idx].copy()
random_test = candidates.iloc[random_test_idx].copy()
group_train = candidates.iloc[group_train_idx].copy()
group_test = candidates.iloc[group_test_idx].copy()
random_overlap = set(random_train['client_hash_id']).intersection(random_test['client_hash_id'])
group_train_clients = set(group_train['client_hash_id'])
group_test_clients = set(group_test['client_hash_id'])
assert group_train_clients.isdisjoint(group_test_clients)
assert random_test[LABEL].nunique() == 2 and group_test[LABEL].nunique() == 2

print(f'Random-row split: {len(random_train):,} train / {len(random_test):,} test rows; {len(random_overlap)} client groups occur on both sides.')
print(f'Client-held-out split: {len(group_train):,} train / {len(group_test):,} test rows; {len(group_train_clients)} train clients / {len(group_test_clients)} test clients (20% of candidate-bearing client groups); overlap = 0.')

def new_model():
    return make_pipeline(
        SimpleImputer(strategy='median'),
        StandardScaler(),
        LogisticRegression(C=1.0, max_iter=2000, random_state=SEED),
    )

random_model = new_model().fit(random_train[FEATURES], random_train[LABEL])
group_model = new_model().fit(group_train[FEATURES], group_train[LABEL])
random_test = random_test.copy()
group_test = group_test.copy()
random_test['model_score'] = random_model.predict_proba(random_test[FEATURES])[:, 1]
group_test['model_score'] = group_model.predict_proba(group_test[FEATURES])[:, 1]

def rank_candidates(frame, score_column):
    ranked = frame.copy()
    tie_breakers = ['clicks_mar', 'avg_position_mar']
    if score_column == 'model_score':
        tie_breakers.insert(0, 'baseline_score')
    sort_columns = [score_column] + tie_breakers
    ascending = [False, False, True] if score_column == 'baseline_score' else [False, False, False, True]
    return ranked.sort_values(sort_columns, ascending=ascending, kind='mergesort').reset_index(drop=True)

def precision_at_k(ranked, k):
    return float(ranked[LABEL].head(k).mean())

comparison_rows = []
for split_name, test_frame in [('Random row', random_test), ('Client held out', group_test)]:
    baseline_test = test_frame.copy()
    baseline_test['baseline_score'] = np.log1p(baseline_test['impressions_mar'])
    test_frame['baseline_score'] = baseline_test['baseline_score']
    ranked_sets = [
        ('ML-07 rule: log impressions', rank_candidates(baseline_test, 'baseline_score')),
        ('ML-08 Logistic Regression', rank_candidates(test_frame, 'model_score')),
    ]
    for method, ranked in ranked_sets:
        scores = ranked['baseline_score'] if method.startswith('ML-07') else ranked['model_score']
        comparison_rows.append({
            'Validation': split_name,
            'Ranking': method,
            'Test items': len(test_frame),
            'Test base rate': test_frame[LABEL].mean(),
            'Precision@10': precision_at_k(ranked, 10),
            'Precision@20': precision_at_k(ranked, 20),
            'Precision@50': precision_at_k(ranked, 50),
            'ROC-AUC': roc_auc_score(test_frame[LABEL], scores),
            'Average precision': average_precision_score(test_frame[LABEL], scores),
        })
comparison = pd.DataFrame(comparison_rows)
print('Same model, ML-07 rule, features, candidate pool, and metrics under each split:')
display(comparison.round(3))
print('Random-row results can benefit from client overlap; client-held-out results test transfer to unseen client groups.')

Random-row split: 31,636 train / 7,909 test rows; 25 client groups occur on both sides.
Client-held-out split: 20,461 train / 19,084 test rows; 24 train clients / 7 test clients (20% of candidate-bearing client groups); overlap = 0.
Same model, ML-07 rule, features, candidate pool, and metrics under each split:


,Validation,Ranking,Test items,Test base rate,Precision@10,Precision@20,Precision@50,ROC-AUC,Average precision
0,Random row,ML-07 rule: log impressions,7909,0.553,0.4,0.25,0.28,0.505,0.556
1,Random row,ML-08 Logistic Regression,7909,0.553,0.8,0.75,0.66,0.505,0.556
2,Client held out,ML-07 rule: log impressions,19084,0.554,0.4,0.30,0.32,0.448,0.536
3,Client held out,ML-08 Logistic Regression,19084,0.554,0.8,0.75,0.72,0.448,0.536


Random-row results can benefit from client overlap; client-held-out results test transfer to unseen client groups.


## 3. Leakage audit

Timeline: **March 2026 GSC features and ML-07 eligibility rule → decision/review ranking → April 2026 impressions outcome.** The outcome is retrospective relative to March, but no April field is used to rank candidates. The outcome definition does use March impressions as its reference denominator; that prior value is a legitimate March-known input, but the evaluation only applies to the March-rule-selected candidate pool.

The main limitation is not direct label leakage in the feature matrix: it is that the target is a one-month observational decline proxy, and the pool is selected by the same March signals before splitting. The model is a ranking of candidates for review, not a causal estimate of refresh impact.

In [6]:
feature_audit = pd.DataFrame([
    {'Field or field family': 'March impressions, clicks, CTR, position, GSC days', 'Timing / role': 'March 2026; available before April outcome', 'Model input': 'Yes', 'Audit result': 'Allowed for this retrospective next-month target'},
    {'Field or field family': 'April impressions / april_decline_label', 'Timing / role': 'April 2026 outcome and label source', 'Model input': 'No', 'Audit result': 'Target/evaluation only; excluded from FEATURES'},
    {'Field or field family': 'Client/content pseudonymous IDs', 'Timing / role': 'Entity keys for grouping and joins', 'Model input': 'No', 'Audit result': 'Grouping only; never predictors'},
    {'Field or field family': 'Trend fields, future windows, action/reason flags', 'Timing / role': 'Label-derived, future, or decision-derived fields', 'Model input': 'No', 'Audit result': 'Excluded from the design'},
    {'Field or field family': 'ML-07 review eligibility', 'Timing / role': 'March-only candidate gate; conditions on signals being evaluated', 'Model input': 'Pool selection only', 'Audit result': 'Not direct label leakage; limits inference to eligible candidates'},
])
display(feature_audit)
forbidden_features = set(ID_COLUMNS + [LABEL, 'impressions_apr', 'trend_pct', 'trend_direction', 'action_label', 'reason_code'])
assert not forbidden_features.intersection(FEATURES)
assert all('mar' in name for name in FEATURES)
assert 'client_hash_id' not in FEATURES and 'content_hash_id' not in FEATURES
print('Leakage checks passed: April/label-derived fields and identifiers are absent from the model feature list.')

,Field or field family,Timing / role,Model input,Audit result
0,"March impressions, clicks, CTR, position, GSC ...",March 2026; available before April outcome,Yes,Allowed for this retrospective next-month target
1,April impressions / april_decline_label,April 2026 outcome and label source,No,Target/evaluation only; excluded from FEATURES
2,Client/content pseudonymous IDs,Entity keys for grouping and joins,No,Grouping only; never predictors
3,"Trend fields, future windows, action/reason flags","Label-derived, future, or decision-derived fields",No,Excluded from the design
4,ML-07 review eligibility,March-only candidate gate; conditions on signa...,Pool selection only,Not direct label leakage; limits inference to ...


Leakage checks passed: April/label-derived fields and identifiers are absent from the model feature list.


### Failure examples from the client-held-out test set

A false positive means the model ranked/classified a candidate as high risk at its 0.5 probability threshold, but the April proxy was 0. A false negative means the probability was below 0.5 even though the proxy was 1. These are disagreements with a coarse observational target, not verified editorial mistakes. The table suppresses identifiers.

In [7]:
group_test = group_test.copy()
group_test['predicted_at_0_5'] = (group_test['model_score'] >= 0.5).astype(int)
group_test['anonymous_rank'] = group_test['model_score'].rank(method='first', ascending=False).astype(int)
false_positive_cases = group_test.loc[
    (group_test['predicted_at_0_5'] == 1) & (group_test[LABEL] == 0)
].sort_values('model_score', ascending=False).head(3)
false_negative_cases = group_test.loc[
    (group_test['predicted_at_0_5'] == 0) & (group_test[LABEL] == 1)
].sort_values('model_score', ascending=True).head(3)
error_examples = []
for error_type, error_frame, explanation in [
    ('False positive', false_positive_cases, 'Candidate met March review rule; April impressions did not decline by the proxy threshold.'),
    ('False negative', false_negative_cases, 'April proxy decline occurred, but the March aggregate signals assigned a low probability.'),
]:
    for _, row in error_frame.iterrows():
        error_examples.append({
            'Error type': error_type,
            'Anonymous model rank': int(row['anonymous_rank']),
            'Model probability': row['model_score'],
            'April proxy': int(row[LABEL]),
            'March impressions': int(row['impressions_mar']),
            'March clicks': int(row['clicks_mar']),
            'March CTR (%)': row['ctr_mar'],
            'March position': row['avg_position_mar'],
            'Observed GSC days': int(row['gsc_days_mar']),
            'Why difficult': explanation,
        })
errors = pd.DataFrame(error_examples)
print(f"At the descriptive 0.5 threshold, {int((group_test['predicted_at_0_5'] != group_test[LABEL]).sum()):,} of {len(group_test):,} grouped-test candidates disagree with the proxy.")
if errors.empty:
    print('No threshold-error examples were available for display in one of the categories.')
else:
    display(errors.round({'Model probability': 3, 'March CTR (%)': 3, 'March position': 2}))
print('These disagreements may also reflect seasonality, query mix, competition, or other unobserved factors absent from the monthly aggregates.')

At the descriptive 0.5 threshold, 8,171 of 19,084 grouped-test candidates disagree with the proxy.


,Error type,Anonymous model rank,Model probability,April proxy,March impressions,March clicks,March CTR (%),March position,Observed GSC days,Why difficult
0,False positive,9,0.856,0,676,0,0.000,0.58,31,Candidate met March review rule; April impress...
1,False positive,10,0.856,0,609,0,0.000,0.67,31,Candidate met March review rule; April impress...
2,False positive,12,0.856,0,607,0,0.000,0.71,31,Candidate met March review rule; April impress...
3,False negative,19074,0.058,1,2063,6,0.291,19.30,22,"April proxy decline occurred, but the March ag..."
4,False negative,19061,0.070,1,550,2,0.364,16.73,22,"April proxy decline occurred, but the March ag..."
5,False negative,19060,0.070,1,655,3,0.458,12.72,22,"April proxy decline occurred, but the March ag..."


These disagreements may also reflect seasonality, query mix, competition, or other unobserved factors absent from the monthly aggregates.


## 4. Claim rewrite

The original claims below are deliberately overconfident examples—not statements supported by this notebook. The rewrite reflects the actual warehouse target and the measured comparison. A higher Precision@50 on one split would be a local top-of-queue result; without corresponding gains in ROC-AUC and average precision, it is not an overall ranking win.

In [8]:
grouped_results = comparison.loc[comparison['Validation'].eq('Client held out')].set_index('Ranking')
grouped_rule = grouped_results.loc['ML-07 rule: log impressions']
grouped_model = grouped_results.loc['ML-08 Logistic Regression']
measured_grouped_rewrite = (
    f"On this client-held-out split, Logistic Regression Precision@50 was {grouped_model['Precision@50']:.3f} versus "
    f"{grouped_rule['Precision@50']:.3f} for the ML-07 rule, while ROC-AUC was {grouped_model['ROC-AUC']:.3f} versus "
    f"{grouped_rule['ROC-AUC']:.3f} and average precision was {grouped_model['Average precision']:.3f} versus "
    f"{grouped_rule['Average precision']:.3f}. This is a split-specific top-of-queue result in the selected candidate pool, "
    "not evidence of an overall ranking gain or guaranteed transfer to future clients."
)
claim_rewrites = pd.DataFrame([
    {
        'Overstated claim': 'The model predicts future Google declines and proves refreshing the recommended pages will restore traffic.',
        'Evidence-aligned rewrite': 'The model ranks ML-07-eligible items by their association with an April month-over-month impression-decline proxy. This retrospective comparison does not establish early warning or the effect of refreshing a page.',
    },
    {
        'Overstated claim': 'Logistic Regression beats the baseline overall and will generalize to every client.',
        'Evidence-aligned rewrite': measured_grouped_rewrite,
    },
    {
        'Overstated claim': 'Low CTR causes pages to lose traffic.',
        'Evidence-aligned rewrite': 'Low March CTR is one condition in the review rule and a model input; the observed association does not show that low CTR caused an April impression decline.',
    },
])
display(claim_rewrites)
for split_name in ['Random row', 'Client held out']:
    subset = comparison.loc[comparison['Validation'].eq(split_name)].set_index('Ranking')
    rule = subset.loc['ML-07 rule: log impressions']
    model = subset.loc['ML-08 Logistic Regression']
    print(
        f"{split_name}: test base rate {model['Test base rate']:.3f}; "
        f"Logistic Regression Precision@50 {model['Precision@50']:.3f} vs rule {rule['Precision@50']:.3f}; "
        f"ROC-AUC {model['ROC-AUC']:.3f} vs {rule['ROC-AUC']:.3f}; "
        f"average precision {model['Average precision']:.3f} vs {rule['Average precision']:.3f}."
    )
print('The random-row split is a diagnostic comparison, not the preferred estimate when client groups repeat.')

,Overstated claim,Evidence-aligned rewrite
0,The model predicts future Google declines and ...,The model ranks ML-07-eligible items by their ...
1,Logistic Regression beats the baseline overall...,"On this client-held-out split, Logistic Regres..."
2,Low CTR causes pages to lose traffic.,Low March CTR is one condition in the review r...


Random row: test base rate 0.553; Logistic Regression Precision@50 0.660 vs rule 0.280; ROC-AUC 0.505 vs 0.505; average precision 0.556 vs 0.556.
Client held out: test base rate 0.554; Logistic Regression Precision@50 0.720 vs rule 0.320; ROC-AUC 0.448 vs 0.448; average precision 0.536 vs 0.536.
The random-row split is a diagnostic comparison, not the preferred estimate when client groups repeat.


## 5. Self-check

- [x] Two findings are accurately described from the assigned March 2026 paper; questions are constructive and methodology-focused.
- [x] Random-row and client-held-out splits use the same model, March feature set, ML-07 candidate pool, April target, and metrics.
- [x] Base rates and split group overlap are reported; identifiers are grouping-only.
- [x] April labels/future fields and trend-derived fields are excluded from model inputs; temporal and candidate-pool limitations are explained.
- [x] Anonymous false-positive and false-negative examples are reviewed.
- [x] Claims use observed, measured, directional, and decision-support language; no causal refresh effect is claimed.
- [x] No credentials, pseudonymous IDs, raw queries, or client names appear in saved outputs.